In [15]:
import numpy as np
import torch
import time
import os
import csv

from datasets import load_dataset
from scipy.stats import pearsonr

from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    TrainerCallback,
)



In [16]:
train_data = load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="train")
val_data = load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="dev")
test_data = load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="test")
print(train_data[1])  # Print the first example to understand its structure

train_df = train_data.to_pandas()
print("Sample data (first 5 rows):")
print(train_df.head())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

{'id': 'eng_train_track_b_00002', 'text': 'This involved swimming a pretty large lake that was over my head.', 'anger': 0, 'disgust': None, 'fear': 2, 'joy': 0, 'sadness': 0, 'surprise': 0}
Sample data (first 5 rows):
                        id                                               text  \
0  eng_train_track_b_00001                       Colorado, middle of nowhere.   
1  eng_train_track_b_00002  This involved swimming a pretty large lake tha...   
2  eng_train_track_b_00003        It was one of my most shameful experiences.   
3  eng_train_track_b_00004  After all, I had vegetables coming out my ears...   
4  eng_train_track_b_00005                        Then the screaming started.   

   anger  disgust  fear  joy  sadness  surprise  
0      0      NaN     1    0        0         1  
1      0      NaN     2    0        0         0  
2      0      NaN     1    0        3         0  
3      0      NaN     0    0        0         0  
4      0      NaN     3    0        1        

In [17]:
TEXT_COL = "text"

EMOTIONS = ["anger", "fear", "joy", "sadness", "surprise"]

In [18]:
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def preprocess(example):
    encoded = tokenizer(
        example[TEXT_COL],
        truncation=True,
        max_length=256,
    )
    encoded["labels"] = [float(example[e]) for e in EMOTIONS]
    return encoded

train_tok = train_data.map(preprocess)
val_tok   = val_data.map(preprocess)
test_tok  = test_data.map(preprocess)

In [19]:
cols = ["input_ids", "attention_mask", "labels"]
train_tok.set_format(type="torch", columns=cols)
val_tok.set_format(type="torch", columns=cols)
test_tok.set_format(type="torch", columns=cols)

In [20]:
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=len(EMOTIONS),
    problem_type="regression"
).to(device)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [21]:
def safe_pearson(x, y):
    r, _ = pearsonr(x, y)
    return 0.0 if np.isnan(r) else float(r)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.asarray(preds)
    labels = np.asarray(labels)

    metrics = {}
    rs = []

    for i, emo in enumerate(EMOTIONS):
        r = safe_pearson(preds[:, i], labels[:, i])
        metrics[f"pearson_{emo}"] = r
        rs.append(r)

    metrics["pearson_mean"] = float(np.mean(rs))
    return metrics

In [24]:
EPOCHS = 3
training_args = TrainingArguments(
    output_dir="roberta_brighter_onlyintensities",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=EPOCHS,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="pearson_mean",
    greater_is_better=True,

    report_to="none",
    fp16=torch.cuda.is_available(),  # speeds up on many Colab GPUs
)

In [25]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

start = time.time()
trainer.train()
end = time.time()

print(f"\nTotal training time: {end - start:.1f} seconds")
print(f"Average time per epoch: {(end - start) / EPOCHS:.1f} seconds")

/tmp/ipython-input-2014210145.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Pearson Anger,Pearson Fear,Pearson Joy,Pearson Sadness,Pearson Surprise,Pearson Mean
1,No log,0.270603,0.748954,0.695305,0.770054,0.782012,0.720748,0.743415
2,0.157500,0.293455,0.725705,0.647248,0.710259,0.808192,0.707413,0.719763
3,0.101400,0.275389,0.732578,0.679327,0.729873,0.803294,0.711160,0.731246



Total training time: 123.5 seconds
Average time per epoch: 41.2 seconds


In [27]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

start = time.time()
trainer.train()
end = time.time()

print(f"\nTotal training time: {end - start:.1f} seconds")
print(f"Average time per epoch: {(end - start) / EPOCHS:.1f} seconds")


Test set evaluation:


eval_loss: 0.26390036940574646
eval_pearson_anger: 0.696506142616272
eval_pearson_fear: 0.7725081443786621
eval_pearson_joy: 0.7788046598434448
eval_pearson_sadness: 0.7701509594917297
eval_pearson_surprise: 0.7146509289741516
eval_pearson_mean: 0.746524167060852
eval_runtime: 5.3806
eval_samples_per_second: 513.886
eval_steps_per_second: 64.306
epoch: 3.0


In [29]:
def predict_intensities(text: str):
    model.eval()

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits.detach().cpu().numpy()[0]  # (5,)

    discrete = np.clip(np.rint(logits), 0, 3).astype(int)

    return {
        EMOTIONS[i]: {"raw": float(logits[i]), "intensity_0_3": int(discrete[i])}
        for i in range(len(EMOTIONS))
    }

print(predict_intensities("I feel so happy and joyful today!"))

{'anger': {'raw': 0.00905609130859375, 'intensity_0_3': 0}, 'fear': {'raw': 0.0284271240234375, 'intensity_0_3': 0}, 'joy': {'raw': 2.96875, 'intensity_0_3': 3}, 'sadness': {'raw': 0.201171875, 'intensity_0_3': 0}, 'surprise': {'raw': 0.0369873046875, 'intensity_0_3': 0}}
